# Notebook 06: Backlash Index & Factor Analysis
This notebook implements the pre-registered Confirmatory Factor Analysis (CFA) using `semopy` on time-demeaned backlash indicators, checks the fit statistics, and provides a robust PCA fallback to construct the final composite backlash index.

### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*

## 1. Library Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import semopy
from sklearn.decomposition import PCA

df = pd.read_csv('../data/processed/state_year_panel.csv')
print(f'Data loaded. Panel size: {df.shape}')

Data loaded. Panel size: (765, 43)


## 2. Time-Demeaning Backlash Indicators
To adjust for nested panel observations and align the measurement model with our state fixed effects regressions, we time-demean the backlash indicators at the state level.

In [2]:
cols_to_demean = ['backlash_media_winsorized', 'svi_common_core_std', 'svi_testing_std', 'svi_opt_out_std']
cols_dm = []

for col in cols_to_demean:
    dm_col = f'{col}_dm'
    df[dm_col] = df[col] - df.groupby('state')[col].transform('mean')
    cols_dm.append(dm_col)

print('Demeaned indicators created:', cols_dm)
print(df[cols_dm].describe())

Demeaned indicators created: ['backlash_media_winsorized_dm', 'svi_common_core_std_dm', 'svi_testing_std_dm', 'svi_opt_out_std_dm']
       backlash_media_winsorized_dm  svi_common_core_std_dm  \
count                  7.650000e+02            7.650000e+02   
mean                  -5.950215e-18           -1.044916e-17   
std                    6.086751e-01            8.964320e-01   
min                   -1.801357e+00           -1.902379e+00   
25%                   -3.856861e-01           -6.085179e-01   
50%                   -7.960353e-02           -1.848467e-01   
75%                    2.916956e-01            4.259250e-01   
max                    3.049991e+00            3.064367e+00   

       svi_testing_std_dm  svi_opt_out_std_dm  
count        7.650000e+02        7.650000e+02  
mean         4.063561e-18        6.675851e-18  
std          5.380903e-01        8.251678e-01  
min         -1.494510e+00       -2.026170e+00  
25%         -2.802418e-01       -1.852101e-01  
50%         

## 3. Confirmatory Factor Analysis (CFA) using semopy
We fit a single-latent-factor model where the latent *Backlash* construct loads on the 4 demeaned indicators. We exclude partisan outcomes and focus strictly on behavioral mobilization variables.

In [3]:
cfa_desc = '''
Backlash =~ backlash_media_winsorized_dm + svi_common_core_std_dm + svi_testing_std_dm + svi_opt_out_std_dm
'''

model = semopy.Model(cfa_desc)
fit_res = model.fit(df)
print('Optimization status:', fit_res.fun)

loadings = model.inspect()
print('\n=== CFA Factor Loadings ===')
print(loadings)

stats = semopy.calc_stats(model)
print('\n=== CFA Fit Statistics ===')
print(stats.T)

Optimization status: 0.2731197802200538

=== CFA Factor Loadings ===
                           lval  op                          rval  Estimate  \
0  backlash_media_winsorized_dm   ~                      Backlash  1.000000   
1        svi_common_core_std_dm   ~                      Backlash -0.104701   
2            svi_testing_std_dm   ~                      Backlash  0.039942   
3            svi_opt_out_std_dm   ~                      Backlash  0.130180   
4                      Backlash  ~~                      Backlash  0.370014   
5  backlash_media_winsorized_dm  ~~  backlash_media_winsorized_dm  0.000000   
6        svi_common_core_std_dm  ~~        svi_common_core_std_dm  0.798567   
7            svi_opt_out_std_dm  ~~            svi_opt_out_std_dm  0.673811   
8            svi_testing_std_dm  ~~            svi_testing_std_dm  0.288623   

   Std. Err    z-value   p-value  
0         -          -         -  
1  0.435511   -0.24041  0.810013  
2  0.167966     0.2378  0.812036  


## 4. Go/No-Go Decision & Fallback Routes
We evaluate fit statistics. If fit is acceptable (CFI > 0.95 and RMSEA < 0.06), we predict the factor scores. Regardless of fit, we also estimate a time-demeaned Principal Component Analysis (PCA) first-component score as a stable algebraic fallback.

In [4]:
# 1. Run PCA fallback
pca = PCA(n_components=1)
df['backlash_pca'] = pca.fit_transform(df[cols_dm])[:, 0]
print('PCA First-Component variance explained:', pca.explained_variance_ratio_[0])

# 2. Extract CFA factor scores if model converged
cfa_scores_extracted = False
try:
    # semopy predict factors returns a DataFrame of predicted factors
    predicted_factors = semopy.predict_factors(model, df)
    if 'Backlash' in predicted_factors.columns:
        df['backlash_cfa'] = predicted_factors['Backlash']
        cfa_scores_extracted = True
        print('CFA factor scores successfully extracted and saved to backlash_cfa.')
except Exception as e:
    print(f'Could not extract CFA factor scores programmatically: {e}')

# 3. Implement the Go/No-Go branching decision
cfi = 0.0
rmsea = 1.0
if 'CFI' in stats.index:
    cfi = stats.loc['CFI'].values[0]
elif 'CFI' in stats.columns:
    cfi = stats['CFI'].values[0]

if 'RMSEA' in stats.index:
    rmsea = stats.loc['RMSEA'].values[0]
elif 'RMSEA' in stats.columns:
    rmsea = stats['RMSEA'].values[0]

print(f'Model Fit Diagnostics: CFI = {cfi:.4f}, RMSEA = {rmsea:.4f}')

if cfi > 0.95 and rmsea < 0.06 and cfa_scores_extracted:
    print('CFA fit is acceptable. Proceeding with backlash_cfa as the primary backlash composite.')
    df['backlash'] = df['backlash_cfa']
else:
    print('CFA fit poor or convergence failed. Falling back to backlash_pca (PCA first-component) for primary analysis.')
    df['backlash'] = df['backlash_pca']

# Define highway control backlash composite (PCA first component of highway indicators)
h_cols = ['highway_backlash_media_winsorized', 'highway_backlash_mass']
h_cols_dm = []
for c in h_cols:
    dm_c = f'{c}_dm'
    df[dm_c] = df[c] - df.groupby('state')[c].transform('mean')
    h_cols_dm.append(dm_c)

pca_h = PCA(n_components=1)
df['highway_backlash'] = pca_h.fit_transform(df[h_cols_dm])[:, 0]
print('Highway control backlash composite created.')

PCA First-Component variance explained: 0.4458006082501111
Could not extract CFA factor scores programmatically: module 'semopy' has no attribute 'predict_factors'
Model Fit Diagnostics: CFI = 0.0396, RMSEA = 0.3680
CFA fit poor or convergence failed. Falling back to backlash_pca (PCA first-component) for primary analysis.
Highway control backlash composite created.


## 5. Save the Updated Panel Dataset
We save the updated panel (containing the new `backlash`, `backlash_pca`, `backlash_cfa`, and `highway_backlash` columns) back to disk.

In [5]:
df.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Panel dataset updated and saved to data/processed/state_year_panel.csv.')

Panel dataset updated and saved to data/processed/state_year_panel.csv.
